In [2]:
%load_ext cudf.pandas

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [3]:
import pickle

# Load the model back
with open('/kaggle/input/datasets/ashura369/my-dataset/df_1.kl', 'rb') as f:
    df = pickle.load(f)

In [4]:
df.head(10)

,id,name,date,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
0,570306133677760513,cairdin,2015-02-24,Tuesday,11,Eastern_Time,Virgin_America,What said.,Flight Booking Problems,0.00000,0,neutral
1,570301130888122368,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,plus you've added commercials to the experienc...,Customer Service Issue,0.00000,0,positive
2,570301083672813571,yvonnalynn,2015-02-24,Tuesday,11,Central_Time,Virgin_America,I didn't today... Must mean I need to take ano...,Flight Attendant Complaints,0.00000,0,neutral
3,570301031407624196,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,"it's really aggressive to blast obnoxious ""ent...",Bad Flight,0.70330,0,negative
4,570300817074462722,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,and it's a really big bad thing about it,Can't Tell,1.00000,0,negative
5,570300767074181121,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,seriously would pay $30 a flight for seats tha...,Can't Tell,0.68420,0,negative
6,570300616901320704,cjmcginnis,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,"yes, nearly every time I fly VX this “ear worm...",Customer Service Issue,0.00000,0,positive
7,570300248553349120,pilot,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,Really missed a prime opportunity for Men With...,Customer Service Issue,0.00000,0,neutral
8,570299953286942721,dhepburn,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,"Well, I didn't…but NOW I DO! :-D",Customer Service Issue,0.00000,0,positive
9,570295459631263746,YupitsTate,2015-02-24,Tuesday,10,Eastern_Time,Virgin_America,"it was amazing, and arrived an hour early. You...",Bad Flight,0.23895,0,positive


In [5]:
df = df.drop(columns=['id','name','date'])

In [6]:
df['reason'] = df['reason'].str.replace(" ", "_")

In [7]:
df.head(5)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
0,Tuesday,11,Eastern_Time,Virgin_America,What said.,Flight_Booking_Problems,0.0000,0,neutral
1,Tuesday,11,Pacific_Time,Virgin_America,plus you've added commercials to the experienc...,Customer_Service_Issue,0.0000,0,positive
2,Tuesday,11,Central_Time,Virgin_America,I didn't today... Must mean I need to take ano...,Flight_Attendant_Complaints,0.0000,0,neutral
3,Tuesday,11,Pacific_Time,Virgin_America,"it's really aggressive to blast obnoxious ""ent...",Bad_Flight,0.7033,0,negative
4,Tuesday,11,Pacific_Time,Virgin_America,and it's a really big bad thing about it,Can't_Tell,1.0000,0,negative


## **Using `MinMaxScaler` on hour, and `StandardScaler` on reason_confidence, and retweets**

In [8]:
data = df.copy()
data.sample(5)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
3615,Wednesday,18,Pacific_Time,United,Now do the right thing and reinstate the ticke...,Cancelled_Flight,0.3367,0,negative
4522,Monday,20,Pacific_Time,Southwest,👏👏👏 on that Late Flightst ad. Makes me happy t...,Late_Flight,0.0000,0,positive
8838,Tuesday,14,Eastern_Time,Delta,we don't need anybody else!,Late_Flight,0.0000,0,positive
587,Tuesday,8,Pacific_Time,United,mobile apps need construction from the ground ...,Can't_Tell,0.3443,0,negative
10934,Friday,6,Quito,US_Airways,I did not want to hang out on your plane all d...,Late_Flight,0.3486,0,negative


In [9]:
data.select_dtypes(include='object').head()

,day_name,timezone,airlines,feedback,reason,sentiment
0,Tuesday,Eastern_Time,Virgin_America,What said.,Flight_Booking_Problems,neutral
1,Tuesday,Pacific_Time,Virgin_America,plus you've added commercials to the experienc...,Customer_Service_Issue,positive
2,Tuesday,Central_Time,Virgin_America,I didn't today... Must mean I need to take ano...,Flight_Attendant_Complaints,neutral
3,Tuesday,Pacific_Time,Virgin_America,"it's really aggressive to blast obnoxious ""ent...",Bad_Flight,negative
4,Tuesday,Pacific_Time,Virgin_America,and it's a really big bad thing about it,Can't_Tell,negative


In [10]:
data.dtypes[data.dtypes == 'object'].reset_index()

,index,0
0,day_name,object
1,timezone,object
2,airlines,object
3,feedback,object
4,reason,object
5,sentiment,object


In [11]:
data.select_dtypes(include='object').nunique()

day_name         7
timezone        78
airlines         6
feedback     14340
reason          10
sentiment        3
dtype: int64

## Making a function to use tokenization and lemmatization

In [12]:
!pip install cleantext

In [13]:
import nltk
from nltk.tokenize import word_tokenize

from nltk.corpus import stopwords
stop_words = stopwords.words('english')

from nltk.stem import WordNetLemmatizer
lmt = WordNetLemmatizer()

from cleantext import clean

In [14]:
def transform(txt):
    txt = clean(txt, lowercase=True, punct=True)
    txt = word_tokenize(txt)

    temp = [lmt.lemmatize(word, pos='v') for word in txt if word not in stop_words]

    temp2 = temp[:]
    temp2 = " ".join(temp2)

    return temp2

In [15]:
transform('Hello this the God King, playing and singing songs of liberation')

'hello god king play sing songs liberation'

In [16]:
data.select_dtypes(include='object').nunique()

day_name         7
timezone        78
airlines         6
feedback     14340
reason          10
sentiment        3
dtype: int64

In [17]:
data['day_name'] = data['day_name'].apply(transform)
data['timezone'] = data['timezone'].apply(transform)
data['airlines'] = data['airlines'].apply(transform)
data['feedback'] = data['feedback'].apply(transform)
data['reason'] = data['reason'].apply(transform)

In [18]:
data.sample(10)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
11306,wednesday,17,easterntime,usairways,finally rectify flight situation thank,flightattendantcomplaints,0.0000,0,positive
10100,sunday,8,athens,usairways,chocolate flight please melt httptcojjdosfyibm,customerserviceissue,0.0000,0,neutral
10723,friday,16,quito,usairways,next plane break seat row another hour delay mad,lateflight,0.6633,0,negative
10249,sunday,3,centraltime,usairways,worst experience cancel flightled flight vouch...,cancelledflight,1.0000,0,negative
8396,thursday,7,quito,delta,okay awesome thank,customerserviceissue,0.0000,0,positive
8797,tuesday,16,easterntime,delta,b6 619 morning boston beautiful san diego real...,badflight,0.6799,0,negative
6704,tuesday,8,mountaintime,southwest,try fly nashville tomorrow look,customerserviceissue,0.0000,0,neutral
8732,tuesday,19,easterntime,delta,6 hour delay suppose land 9pm 3am still board ...,lateflight,1.0000,0,negative
9961,sunday,12,centraltime,usairways,pretty ridiculous phx sky harbor 4 employees w...,longlines,1.0000,0,negative
12843,monday,15,sydney,american,bag check cancel flightled flight 362 arrive m...,customerserviceissue,0.3536,0,negative



## Using label encoder on sentiment

In [19]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data[['sentiment']] = le.fit_transform(data[['sentiment']]) 

In [20]:
le.classes_

array(['negative', 'neutral', 'positive'], dtype=object)

In [21]:
data.head(5)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
0,tuesday,11,easterntime,virginamerica,say,flightbookingproblems,0.0000,0,1
1,tuesday,11,pacifictime,virginamerica,plus youve add commercials experience tacky,customerserviceissue,0.0000,0,2
2,tuesday,11,centraltime,virginamerica,didnt today must mean need take another trip,flightattendantcomplaints,0.0000,0,1
3,tuesday,11,pacifictime,virginamerica,really aggressive blast obnoxious entertainmen...,badflight,0.7033,0,0
4,tuesday,11,pacifictime,virginamerica,really big bad thing,canttell,1.0000,0,0


## Using vectorization to convert all the text into vectors

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

#### Using ColumnTransformer to add the needed columns to add into one single column

In [23]:
data['combined_labels'] = data[['day_name','timezone','airlines','reason']].astype(str).agg(' '.join, axis=1)
data.sample(6)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,combined_labels
14126,sunday,17,centraltime,american,need miracle please help flight 1228 get 640 f...,lateflight,1.00000,0,0,sunday centraltime american lateflight
65,monday,14,centraltime,virginamerica,flight 0736 dal dca 224 210pm try check could ...,customerserviceissue,0.00000,0,1,monday centraltime virginamerica customerservi...
1391,sunday,22,pacifictime,unite,actually flight cancel flightled httptcoqf0oc2...,cancelledflight,1.00000,0,0,sunday pacifictime unite cancelledflight
13561,monday,4,pacifictime,american,sure make stand line outside plane isnt ready ...,longlines,1.00000,0,0,monday pacifictime american longlines
10368,saturday,20,quito,usairways,trbl experience 2 hrs tarmac inch snow phl han...,lateflight,0.69950,0,0,saturday quito usairways lateflight
11696,tuesday,14,easterntime,usairways,flight 604 thank,customerserviceissue,0.40293,0,2,tuesday easterntime usairways customerservicei...


In [24]:
processor = ColumnTransformer(
    transformers=[
        ('labels_tf', TfidfVectorizer(ngram_range=(2,2)), 'combined_labels'),
        ('sentence_tf', TfidfVectorizer(ngram_range=(2,2)), 'feedback'),
        ('minmax', MinMaxScaler(), ['hour']),
        ('standard', StandardScaler(), ['reason_confidence', 'retweets'])
    ],
    remainder='drop'            # will drop rest of the unnecessary columns
)


In [25]:
x = processor.fit_transform(data)
print(len(x.toarray()))
x.toarray()

14640


array([[ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
         0.49054576, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799]])

In [26]:
y = data['sentiment'].values
y

array([1, 2, 1, ..., 1, 0, 1])

# Training the model

In [27]:
from sklearn.model_selection import train_test_split as ttt

# 1. Standard dataset split
x_train, x_test, y_train, y_test = ttt(x, y, test_size=0.3, random_state=42)

# 2. Extract into clean NumPy arrays (stored on CPU RAM to support clean KFold indexing)
if hasattr(x_train, "toarray"):
    x_train = x_train.toarray()
    x_test = x_test.toarray()
else:
    x_train = np.array(x_train)
    x_test = np.array(x_test)

# Convert labels to matching NumPy format
y_train = np.array(y_train)
y_test = np.array(y_test)

print("Training array setup on CPU safely. Shape:", x_train.shape)

Training array setup on CPU safely. Shape: (10248, 81510)


# Using Optuna to find the best model 

In [33]:
import optuna
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split as ttt

def model(trial):
    classifier_name = trial.suggest_categorical('classifier', ['RandomForestClassifier', 'ExtraTreesClassifier'])

    x_tr, x_val, y_tr, y_val = ttt(x_train, y_train, test_size=0.2, random_state=42)

    if classifier_name == 'RandomForestClassifier':
        n_estimators = trial.suggest_int('n_estimators', 50, 500, step=50)
        max_depth = trial.suggest_int('max_depth', 5, 50, step=5)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        clf = cuRF(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        clf.fit(x_tr, y_tr)

    elif classifier_name == 'ExtraTreesClassifier':
        n_estimators = trial.suggest_int('n_estimators', 50, 500, step=50)
        max_depth = trial.suggest_int('max_depth', 5, 50, step=5)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        clf = ExtraTreesClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
            n_jobs=-1
        )
        clf.fit(x_tr, y_tr)

    preds = clf.predict(x_val)
    return accuracy_score(y_val, preds)



In [34]:
study = optuna.create_study(direction='maximize')
study.optimize(model, n_trials=20)

best_params = study.best_params
classifier_name = best_params.pop('classifier')



[I 2026-05-26 16:44:51,437] A new study created in memory with name: no-name-91c39d34-1c06-46ad-8c03-c2853e97b7e8
[W 2026-05-26 16:44:52,931] Trial 0 failed with parameters: {'classifier': 'RandomForestClassifier', 'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 4, 'min_samples_leaf': 3} because of the following error: NameError("name 'cuRF' is not defined").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1268/3105254523.py", line 16, in model
    clf = cuRF(
          ^^^^
NameError: name 'cuRF' is not defined
[W 2026-05-26 16:44:52,934] Trial 0 failed with value None.


NameError: name 'cuRF' is not defined

In [ ]:
if classifier_name == 'RandomForestClassifier':
    final_model = cuRF(**best_params, random_state=42)
elif classifier_name == 'ExtraTreesClassifier':
    final_model = ExtraTreesClassifier(**best_params, random_state=42, n_jobs=-1)

final_model.fit(x_train, y_train)
final_preds = final_model.predict(x_test)
final_accuracy = accuracy_score(y_test, final_preds)

print("Best Parameters:", study.best_params)
print("Final Test Accuracy:", final_accuracy)